In [1]:
import random
from pathlib import Path

import numpy as np
import torch

DATA_DIR = Path("output")

OUTPUT_DIR = Path("model_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42


def set_seed(seed=RANDOM_STATE):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


set_seed()

In [2]:
dataset_path = Path("output/dataset_winsize1h_where.csv")

In [3]:
import json

import pandas as pd

where_df = pd.read_csv(
    dataset_path,
    dtype={"fold_id": "int64"},
    converters={
        "window": json.loads,
        "label": json.loads,
    },
)
where_df = where_df.loc[~where_df["is_bg"]].drop(columns="is_bg")

where_df.head()

,fold_id,window,label
0,2,"[{'from_zone_index': 16, 'to_zone_index': 16, ...","{'from_zone_index': 12, 'to_zone_index': 12}"
1,2,"[{'from_zone_index': 9, 'to_zone_index': 9, 'o...","{'from_zone_index': 17, 'to_zone_index': 17}"
5,3,"[{'from_zone_index': 0, 'to_zone_index': 0, 'o...","{'from_zone_index': 8, 'to_zone_index': 8}"
8,-1,"[{'from_zone_index': 4, 'to_zone_index': 4, 'o...","{'from_zone_index': 2, 'to_zone_index': 2}"
9,4,"[{'from_zone_index': 8, 'to_zone_index': 5, 'o...","{'from_zone_index': 2, 'to_zone_index': 2}"


In [4]:
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

OUTAGE_TYPE_TO_ID = {
    "Planned": 0,
    "Auto": 1,
}


class WhereOutageDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        window = row["window"]
        label = row["label"]

        outage_type = torch.tensor(
            [OUTAGE_TYPE_TO_ID[event["outage_type"]] for event in window],
            dtype=torch.long,
        )
        from_zone_indices = torch.tensor(
            [event["from_zone_index"] for event in window],
            dtype=torch.long,
        )
        to_zone_indices = torch.tensor(
            [event["to_zone_index"] for event in window],
            dtype=torch.long,
        )
        time_interval_index = torch.tensor(
            [event["time_interval_index"] for event in window],
            dtype=torch.long,
        )
        target = torch.tensor(
            [label["from_zone_index"], label["to_zone_index"]],
            dtype=torch.long,
        )

        return {
            "outage_type": outage_type,
            "from_zone_indices": from_zone_indices,
            "to_zone_indices": to_zone_indices,
            "time_interval_index": time_interval_index,
            "target": target,
        }


TRAIN_FOLDS = [0, 1, 2, 3, 4]
TEST_FOLDS = [-1]

train_where_dataset = WhereOutageDataset(
    where_df[where_df["fold_id"].isin(TRAIN_FOLDS)]
)
test_where_dataset = WhereOutageDataset(where_df[where_df["fold_id"].isin(TEST_FOLDS)])

In [5]:
for i, sample_dict in enumerate(train_where_dataset):
    print(sample_dict)
    if i == 10:
        break

{'outage_type': tensor([0]), 'from_zone_indices': tensor([16]), 'to_zone_indices': tensor([16]), 'time_interval_index': tensor([2]), 'target': tensor([12, 12])}
{'outage_type': tensor([0]), 'from_zone_indices': tensor([9]), 'to_zone_indices': tensor([9]), 'time_interval_index': tensor([1]), 'target': tensor([17, 17])}
{'outage_type': tensor([0]), 'from_zone_indices': tensor([0]), 'to_zone_indices': tensor([0]), 'time_interval_index': tensor([1]), 'target': tensor([8, 8])}
{'outage_type': tensor([0]), 'from_zone_indices': tensor([8]), 'to_zone_indices': tensor([5]), 'time_interval_index': tensor([2]), 'target': tensor([2, 2])}
{'outage_type': tensor([0, 0, 0]), 'from_zone_indices': tensor([1, 2, 2]), 'to_zone_indices': tensor([16,  2,  2]), 'time_interval_index': tensor([0, 0, 0]), 'target': tensor([2, 2])}
{'outage_type': tensor([1, 0]), 'from_zone_indices': tensor([9, 8]), 'to_zone_indices': tensor([9, 8]), 'time_interval_index': tensor([2, 2]), 'target': tensor([21, 21])}
{'outage_ty

In [6]:
from tqdm import tqdm as tdqm

num_time_intervals = 0
for sample_dict in tdqm(train_where_dataset):
    num_time_intervals = max(
        num_time_intervals, sample_dict["time_interval_index"].max().item() + 1
    )
print(f"{num_time_intervals = }")

100%|██████████| 2967/2967 [00:00<00:00, 11565.91it/s]

num_time_intervals = 5


In [7]:
ZONE_EMBEDDING_DIM = 256
OUTAGE_TYPE_EMBEDDING_DIM = 64
TIME_INTERVAL_EMBEDDING_DIM = 64
TRANSFORMER_DIM = 512
DROPOUT = 0.15

In [ ]:
import torch.nn as nn

num_zones = int(
    max(
        where_df["label"]
        .map(lambda x: max(x["from_zone_index"], x["to_zone_index"]))
        .max(),
        max(
            max(event["from_zone_index"], event["to_zone_index"])
            for window in where_df["window"]
            for event in window
        ),
    )
    + 1
)
num_outage_types = len(OUTAGE_TYPE_TO_ID)


class WhereTransformer(nn.Module):
    """ "Vanilla transformer model"""

    def __init__(
        self,
        num_zones,
        num_outage_types,
        zone_embedding_dim=ZONE_EMBEDDING_DIM,
        outage_type_embedding_dim=OUTAGE_TYPE_EMBEDDING_DIM,
        time_interval_embedding_dim=TIME_INTERVAL_EMBEDDING_DIM,
        transformer_dim=TRANSFORMER_DIM,
        dropout=DROPOUT,
    ):
        super().__init__()
        self.num_zones = num_zones
        self.zone_embedding = nn.Embedding(num_zones, zone_embedding_dim)
        self.outage_type_embedding = nn.Embedding(
            num_outage_types,
            outage_type_embedding_dim,
        )
        self.time_interval_embedding = nn.Embedding(
            num_time_intervals,
            time_interval_embedding_dim,
        )
        self.input_fc = nn.Linear(
            2 * zone_embedding_dim
            + outage_type_embedding_dim
            + time_interval_embedding_dim,
            transformer_dim,
        )
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=transformer_dim,
            nhead=transformer_dim // 64,
            dim_feedforward=transformer_dim * 4,
            dropout=dropout,
            batch_first=True,
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=1,
        )
        self.projection = nn.Linear(transformer_dim, 2 * num_zones)

    def forward(
        self,
        from_zone_indices,
        to_zone_indices,
        outage_type,
        time_interval_index,
    ):
        from_zone_emb = self.zone_embedding(from_zone_indices)
        to_zone_emb = self.zone_embedding(to_zone_indices)
        outage_type_emb = self.outage_type_embedding(outage_type)
        time_interval_emb = self.time_interval_embedding(time_interval_index)

        x = torch.cat(
            [from_zone_emb, to_zone_emb, outage_type_emb, time_interval_emb],
            dim=-1,
        )
        x = self.input_fc(x)

        x = self.transformer_encoder(x)

        # AVerage pooling over the time dimension (dim=1)
        # pooled = x.mean(dim=1)
        # Use the last time step's output for pooling
        pooled = x[:, -1, :]

        logits = self.projection(pooled)
        return logits.view(-1, 2, self.num_zones)


class SelfAttentivePooling(nn.Module):
    def __init__(self, input_dim, attention_dim=None, dropout=DROPOUT):
        super().__init__()
        attention_dim = attention_dim or input_dim
        self.attention = nn.Sequential(
            nn.Linear(input_dim, attention_dim),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(attention_dim, 1),
        )

    def forward(self, x):
        attention_scores = self.attention(x).squeeze(-1)
        attention_weights = torch.softmax(attention_scores, dim=1).unsqueeze(-1)
        return (x * attention_weights).sum(dim=1)


class WhereTransformerSAP(nn.Module):
    def __init__(
        self,
        num_zones,
        num_outage_types,
        zone_embedding_dim=ZONE_EMBEDDING_DIM,
        outage_type_embedding_dim=OUTAGE_TYPE_EMBEDDING_DIM,
        time_interval_embedding_dim=TIME_INTERVAL_EMBEDDING_DIM,
        transformer_dim=TRANSFORMER_DIM,
        dropout=DROPOUT,
    ):
        super().__init__()
        self.num_zones = num_zones
        self.zone_embedding = nn.Embedding(num_zones, zone_embedding_dim)
        self.outage_type_embedding = nn.Embedding(
            num_outage_types, outage_type_embedding_dim
        )
        self.time_interval_embedding = nn.Embedding(
            num_time_intervals, time_interval_embedding_dim
        )
        self.input_fc = nn.Linear(
            2 * zone_embedding_dim
            + outage_type_embedding_dim
            + time_interval_embedding_dim,
            transformer_dim,
        )
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=transformer_dim,
            nhead=transformer_dim // 64,
            dim_feedforward=transformer_dim * 4,
            dropout=dropout,
            batch_first=True,
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=1)
        self.self_attentive_pooling = SelfAttentivePooling(
            transformer_dim, dropout=dropout
        )
        self.projection = nn.Linear(transformer_dim, 2 * num_zones)

    def forward(
        self,
        from_zone_indices,
        to_zone_indices,
        outage_type,
        time_interval_index,
    ):
        x = torch.cat(
            [
                self.zone_embedding(from_zone_indices),
                self.zone_embedding(to_zone_indices),
                self.outage_type_embedding(outage_type),
                self.time_interval_embedding(time_interval_index),
            ],
            dim=-1,
        )
        x = self.input_fc(x)
        x = self.transformer_encoder(x)
        pooled = self.self_attentive_pooling(x)
        logits = self.projection(pooled)
        return logits.view(-1, 2, self.num_zones)

In [ ]:
CHECKPOINT_PATH = (
    OUTPUT_DIR / "where_transformer" / "where_transformer_adjacency_aware_sap_last.pt"
)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
except TypeError:
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

state_dict = checkpoint["model_state_dict"]
checkpoint_num_time_intervals, checkpoint_time_embedding_dim = state_dict[
    "time_interval_embedding.weight"
].shape
if checkpoint_num_time_intervals != num_time_intervals:
    raise ValueError(
        "Checkpoint and dataset use different numbers of time intervals: "
        f"{checkpoint_num_time_intervals} != {num_time_intervals}"
    )

adjacency_where_transformer_sap = WhereTransformerSAP(
    num_zones=checkpoint["num_zones"],
    num_outage_types=checkpoint["num_outage_types"],
    zone_embedding_dim=checkpoint["zone_embedding_dim"],
    outage_type_embedding_dim=checkpoint["outage_type_embedding_dim"],
    time_interval_embedding_dim=checkpoint_time_embedding_dim,
    transformer_dim=checkpoint["transformer_dim"],
    dropout=checkpoint["dropout"],
).to(DEVICE)
adjacency_where_transformer_sap.load_state_dict(state_dict)
adjacency_where_transformer_sap.eval()

model = adjacency_where_transformer_sap
print(f"Loaded adjacency-aware SAP weights from {CHECKPOINT_PATH}")
print(f"Model is ready for interpretation on {DEVICE}.")

Loaded adjacency-aware SAP weights from model_outputs/where_transformer/where_transformer_adjacency_aware_sap_last.pt
Model is ready for interpretation on cuda.
